In [1]:
import json
import pandas as pd
from collections import defaultdict

from src.skill_rules import TIER1_PHRASES, TIER2_EXACT_CASE, TIER3_CUSTOM
from src.models import build_nlp, entity_ruler_extract_skills

In [2]:
# Samla alla fraser och stavningar från samtliga tiers.
# Vi normaliserar till lowercase och tar bort extra blanksteg
# så att exempelvis "Docker" och " docker " upptäcks som dubletter.

def normalize(text):
    return " ".join(text.casefold().split())


occurrences = defaultdict(list)

# Tier 1
for phrase in TIER1_PHRASES:
    occurrences[normalize(phrase)].append(
        ("TIER1_PHRASES", phrase, phrase)
    )

# Tier 2
for skill, spellings in TIER2_EXACT_CASE.items():
    for spelling in spellings:
        occurrences[normalize(spelling)].append(
            ("TIER2_EXACT_CASE", spelling, skill)
        )

# Tier 3
for skill, aliases in TIER3_CUSTOM.items():
    for text in aliases.get("any_case", []):
        occurrences[normalize(text)].append(
            ("TIER3_CUSTOM any_case", text, skill)
        )

    for text in aliases.get("exact_case", []):
        occurrences[normalize(text)].append(
            ("TIER3_CUSTOM exact_case", text, skill)
        )


# Skriv ut alla fraser som förekommer mer än en gång.
duplicates = {
    phrase: entries
    for phrase, entries in occurrences.items()
    if len(entries) > 1
}

if duplicates:
    print(f"Hittade {len(duplicates)} fraser med dubbletter:\n")

    for phrase, entries in sorted(duplicates.items()):
        print(f"Fras: {phrase!r}")

        for tier, spelling, skill in entries:
            print(
                f"  - {tier}: {spelling!r} "
                f"(skill: {skill})"
            )

        print()
else:
    print("Inga dubbletter hittades mellan tiers.")

Inga dubbletter hittades mellan tiers.


In [3]:
# Hitta fraser som är kopplade till fler än en unik skill.

phrase_skills = defaultdict(set)

for phrase, entries in occurrences.items():
    for tier, spelling, skill in entries:
        phrase_skills[phrase].add(skill)

conflicting_skills = {
    phrase: skills
    for phrase, skills in phrase_skills.items()
    if len(skills) > 1
}

if conflicting_skills:
    print(
        f"Hittade {len(conflicting_skills)} fraser "
        "som tillhör flera skills:\n"
    )

    for phrase, skills in sorted(conflicting_skills.items()):
        print(f"{phrase!r} -> {sorted(skills)}")
else:
    print("Inga fraser kopplade till flera skills.")

Inga fraser kopplade till flera skills.


### Bygger pipelinen

In [4]:
# Building pipeline
nlp = build_nlp()

ruler = nlp.get_pipe("entity_ruler")

# Show how many patterns
print(len(ruler.patterns), "patterns")

1156 patterns


### Litet test

In [5]:
ad = """
Vi söker en konsult med erfarenhet av kravhantering och systemanalys.
Du kan AWS, docker och Azure. Kunskap om SAP är meriterande.
"""

print(entity_ruler_extract_skills(ad, nlp))

['Amazon Web Services', 'Docker', 'Kravhantering', 'SAP', 'Systemanalys']


### Entity ruler på train set

In [6]:
with open("../data/train_set.json", "r", encoding="utf-8") as file:
    ads = json.load(file)

df = pd.DataFrame(ads)

In [7]:
df["skills"] = df["description"].apply(lambda text: entity_ruler_extract_skills(text, nlp))

In [8]:
df[["description", "skills"]].head()

,description,skills
0,Har du erfarenhet av Dynamics 365 och/eller AX...,"[Engelska, Microsoft Dynamics 365, Svenska]"
1,"Omvärlden förändras i snabb takt. Ny teknik, n...","[Digitalisering, Dokumentation, Förändringsled..."
2,Är du en analytisk och lösningsorienterad pers...,[Artificial Intelligence / Artificiell intelli...
3,Vill du arbeta med den senaste tekniken och hj...,"[Configuration Management / CM, Engelska, Micr..."
4,Lantmäteriet är den myndighet som kartlägger S...,"[Driva workshops, Engelska, Svenska]"


In [9]:
# How many skills were found per ad?
df["n_skills"] = df["skills"].str.len()
print(df["n_skills"].describe())
print()

# Most common skills across all ads
skill_counts = df["skills"].explode().value_counts()
print(skill_counts.head(30))

count    3081.000000
mean        4.461863
std         2.824508
min         0.000000
25%         3.000000
50%         4.000000
75%         6.000000
max        21.000000
Name: n_skills, dtype: float64

skills
Svenska                                                   1934
Engelska                                                  1347
Agile                                                      761
Digitalisering                                             609
Dokumentation                                              534
Automationsteknik                                          388
Artificial Intelligence / Artificiell intelligens / AI     378
SQL                                                        277
Dataskyddsförordningen / GDPR, erfarenhet                  273
Verksamhetsutveckling                                      259
Kravhantering                                              256
Python                                                     221
Ledning                              

In [10]:
# Checks characters before and after a specific detected skill
skill = "Cybersäkerhet"
skill_ads = df[df["skills"].apply(lambda skills: skill in skills)]

for _, row in skill_ads.head(5).iterrows():
    print("TITLE:", row["title"])
    text = row["description"]

    for ent in nlp(text).ents:
        if ent.ent_id_ == skill:
            start = ent.start_char
            print(ent.text, "->", ent.ent_id_)
            print(text[max(0, start - 50): start + 50])
            print("-" * 80)

TITLE: IT-Arkitekt inom IAM
cybersäkerhet -> Cybersäkerhet
 I rollen kommer fokus vara på identitetssäkerhet/cybersäkerhet, behörighetskontroll och livscykelha
--------------------------------------------------------------------------------
TITLE: Solution Architect
cybersäkerhet -> Cybersäkerhet
i söker en mycket erfaren Solution Architect inom cybersäkerhet till ett tekniskt komplext uppdrag m
--------------------------------------------------------------------------------
TITLE: Systemingenjör Härjeåns
cybersäkerhet -> Cybersäkerhet
en är det meriterande. Du behöver ha intresse för cybersäkerhet och mjukvaruutveckling samt goda kun
--------------------------------------------------------------------------------
TITLE: Senior Solution Architect  sökes till Iver
cybersäkerhet -> Cybersäkerhet
serade IT-tjänster med spetskompetens inom drift, cybersäkerhet, cloud och innovation. Våra tjänster
--------------------------------------------------------------------------------
TITLE: IT-Ark

## Framtagning och strukturering av kompetenslistan
**1. Jobtechs taxonomi som utgångspunkt**   
Vi började med att använda JobTechs taxonomi som grund för vår lista över kompetenser. Syftet var att utgå från en befintlig och strukturerad lista med kompetenser som kunde användas som baseline för vår kompetensmatchning. 

**2. Komplettering av kompetenslistan**    
Därefter delade vi upp arbetet mellan gruppmedlemmarna. Vi gick igenom olika rubriker (Headlines) i taxonomin, exempelvis Datorspråk, Ramverk, Certifikat/Licenser, och kompletterade listan med ca 5 kompetenser per rubrik. På så sätt utökade vi den ursprungliga listan med fler kompetenser som vi ville kunna identifiera i jobbannonserna. 

**3. Indelning i tre tiers**   
När kompetenslistan hade sammanställts delade vi in matchningsreglerna i tre olika tiers för att hantera olika typer av kompetensnamn och stavningar:
- **Tier 1 - Phrases, any case**: Taxonomins ursprungliga kompetensnamn användes som fraser och matchades oavsett stora eller små bokstäver. 
- **Tier 2 - Exact case**: Korta eller tvetydliga ord placerades i denna kategori för att endast matchas med exakt angivet skiftläge. Detta minskar risken för oönskade träffar. 
- **Tier 3 - Custom aliases**: Egna alias, synonymer och alternativa stavningar lades till för att fånga fler sätt att uttrycka kompetenser i olika jobbannonser. Dessa delades upp i regler som matchas oavsett skiftläge respektive regler som använder exakt skiftläge.   

**4. Identifiering och hantering av dubbletter**
Slutligen genomförde vi en kontroll av kompetenslistan för att identifiera dubbletter både inom samma tier och mellan olika tiers. Vi undersökte även om samma fras förekom som alias eller synonym under flera olika kompetensnamn. Kontrollen identifierade bland annat identiska fraser, alternativa skrivsätt och fraser som var kopplade till flera kompetenser.

Exempel på dubletter som identifierades:   

```bash
Fras: 'agila arbetsmetoder'   
TIER3_CUSTOM any_case: 'Agila Arbetsmetoder' (skill: Agila Arbetsmetoder)    
TIER3_CUSTOM any_case: 'Agila arbetsmetoder' (skill: Agile)  
```

Vi hittade även 9 fraser som tillhör flera skills:

```bash
'agila arbetsmetoder' -> ['Agila Arbetsmetoder', 'Agile']   
'agila metoder' -> ['Agila Arbetsmetoder', 'Agile']   
'agile' -> ['Agila Arbetsmetoder', 'Agile']   
'agilt' -> ['Agila Arbetsmetoder', 'Agile']   
'cybersäkerhet' -> ['Cybersäkerhet', 'Cybersäkerhet, nätverk']    
'objektorienterad analys och design' -> ['Objektorienterad analys och design', 'Objektorienterad metodik']    
'ooad' -> ['Objektorienterad analys och design', 'Objektorienterad metodik']   
'ood' -> ['Objektorienterad design', 'Objektorienterad metodik']   
'systemanalys' -> ['Systemanalys', 'Systemutredning']
``` 
   
De identifierade fallen granskades och hanterades manuellt. Identiska dubbletter togs bort, medan relevanta alternativa stavningar och synonymer behölls där de kunde bidra till att fånga fler kompetensomnämnanden. När samma fras var kopplad till flera kompetensnamn justerades reglerna manuellt för att undvika oönskade dubbla kopplingar.    

Resultatet blev en mer strukturerad och kvalitetssäkrad kompetenslista med matchningsregler som är utformade för att identifiera så många relevanta kompetensomnämnanden som möjligt i jobbannonserna. 

